# Thesis Code Walkthrough — Reverse-Engineering Human Latency in VR

**Author:** Selim Can Mutlu (2119179)  
**Programme:** BSc Cognitive Science & Artificial Intelligence — Tilburg University  
**Supervisor:** dr. Ifigeneia Mavridou  
**Dataset:** Mavridou et al. (2025), *Affect recognition in immersive room-scale environments*, IEEE Access

---

## How to read this notebook

This notebook is the **entry point** for the code submission. It walks through every `.py` file in this folder, in the order in which they should be run, explaining **what each one does** and **why** it exists at that point in the pipeline.

The heavy training scripts are **not re-executed here** — that would take hours of GPU time. Instead, every section imports the relevant module (so you can see the source on screen with `??`) and then loads the **pre-computed result** from `results/` to show what the script produced when it last ran on the full cohort.

If you want to reproduce a result, the **"Reproduce"** section at the bottom gives the exact shell command for each script.

## Research questions

| RQ | Question | Outcome |
|---|---|---|
| **Main** | Can a deep model (BiLSTM) learn the variable EMG→valence delay on its own? | **No** — the cross-subject signal ceiling caps performance regardless of architecture. |
| **SQ1** | Does the BiLSTM beat a Random Forest with the static 200 ms lag from Mavridou et al. (2025)? | **No** — RF wins on 12 of 13 test subjects (Wilcoxon $p = 0.0007$). |
| **SQ2** | Does the empirically measured delay vary with stimulus saliency? | **Directional trend, not BH-significant.** Delay B is +141 ms longer for positive vs negative scenes ($p_{\text{raw}} = 0.022$); none of the 44 modulation tests survive Benjamini–Hochberg correction. |

**Headline empirical finding** (the contribution that anchors the thesis):

$$\text{Delay A} = 227\,\text{ms} \qquad \text{Delay C} = 1{,}717\,\text{ms} \qquad \text{Delay B} = \text{C} - \text{A} = 1{,}379\,\text{ms} \approx 1.4\,\text{s}$$

The 200 ms alignment heuristic used in prior work captures only ~12% of the true cognitive–motor gap.

## Pipeline map — script execution order

```
 ┌─────────────────────────────────────────────────────────────────────────┐
 │  STAGE 1 — PREPROCESSING                                                │
 │                                                                         │
 │   run_pipeline.py                                                       │
 │     raw .txt + .json + rating ─► event_windows_all.pkl                  │
 │     (bandpass → Hampel → rectify → envelope; event onsets;              │
 │      saliency taxonomy; ±2/+5 s windowing; 10 Hz downsample)            │
 │                                                                         │
 │   extract_spectral_features.py                                          │
 │     event_windows_all.pkl ─► event_windows_spectral.pkl                 │
 │     (459 spectral + advanced-time-domain features per event)            │
 └─────────────────────────────────────────────────────────────────────────┘
                                  │
 ┌─────────────────────────────────────────────────────────────────────────┐
 │  STAGE 2 — DELAY EXTRACTION (the headline 1.4 s result)                 │
 │                                                                         │
 │   estimate_delays.py                                                    │
 │     event_windows_spectral.pkl + raw 1000 Hz EMG ─► delays_per_event.csv│
 │     (rule-based; threshold + slope rules; literature validity gates)    │
 └─────────────────────────────────────────────────────────────────────────┘
                                  │
 ┌─────────────────────────────────────────────────────────────────────────┐
 │  STAGE 3 — STATISTICS                                                   │
 │                                                                         │
 │   analyze_delays.py    delays_per_event.csv ─► plots + summary          │
 │   analyze_modulation.py + metadata Excel    ─► modulation_stats.csv     │
 └─────────────────────────────────────────────────────────────────────────┘
                                  │
 ┌─────────────────────────────────────────────────────────────────────────┐
 │  STAGE 4 — MODELS                                                       │
 │                                                                         │
 │   train_lstm_v4_full.py       RF vs BiLSTM, continuous valence (SQ1)    │
 │   train_lstm_multiband.py     earlier multi-band variant (Figure 5)     │
 │   train_classification.py     binary valence-direction classifier       │
 │   train_spectral.py           saliency AUC = 0.729 (459-feature bank)   │
 │   train_svm_variations.py     SVM × 4 timing windows × 3 feature sets   │
 │   train_spectral_slope.py     exploratory shape-feature future work     │
 └─────────────────────────────────────────────────────────────────────────┘
```

**Why this order?**

1. Stage 1 produces the cached event windows that every later script consumes. It is the most expensive step (full bandpass + Hampel + envelope on 1000 Hz raw streams across 116 participants), so it is run once and cached.
2. Stage 2 needs the windows from Stage 1 **and** the raw 1000 Hz EMG (it reloads it for the 1 ms-precision Delay A detection). It produces `delays_per_event.csv`, which is the foundation for everything that follows.
3. Stage 3 reads `delays_per_event.csv`. The modulation analysis additionally joins the participant-metadata Excel (questionnaires, demographics).
4. Stage 4 reads the cached spectral windows from Stage 1 (Stage 2 is independent of Stage 4 — they share Stage 1 input but do not depend on each other).

The notebook below walks through each script in this order.

In [ ]:
# Setup — paths and results loader. Run this cell once before the rest.
from pathlib import Path
import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

# All paths are relative to this notebook's folder (submission/code/).
HERE = Path('.').resolve()
RESULTS = HERE / 'results'
PLOTS = HERE.parent / 'thesis' / 'plots'

print('Notebook location :', HERE)
print('Results folder    :', RESULTS, ' exists:', RESULTS.exists())
print('Plots folder      :', PLOTS,   ' exists:', PLOTS.exists())

---
# Stage 1 — Preprocessing

## 1A. `run_pipeline.py` — raw streams → event windows

**What it does.** Ingests three raw files per (participant, scene):

| File | Sampling rate | Contents |
|---|---|---|
| `*_raw.txt` | 1000 Hz | 9 facial-EMG channels + 2 HR + 9 IMU + 3 accel + magnetometer + gyro |
| `*.json` | per-event | EventID list, computed valence/arousal, expression intensities |
| `Rating_Scene_*.txt` | continuous | Joystick valence and arousal in $[-1, +1]$ |

Then it:

1. **Merges** the three streams on the shared `frame` index (inner-join with the JSON, `merge_asof` with the rating stream at 50-frame tolerance).
2. **Detects event onsets** as the first frame at which a new `EventID` enters the active event-set (set difference against the previous frame — this is more robust than equality because stimuli often overlap).
3. **Classifies each event** as high- or low-saliency using a hand-built EventID taxonomy derived from the Mavridou et al. (2025) supplementary materials.
4. **Computes the EMG envelope** per channel via the four-stage pipeline below.
5. **Cuts a $\pm$ window** around every event onset ($-2$ s pre, $+5$ s post) and downsamples 1000 Hz → 10 Hz.

**Why a four-stage EMG pipeline?**

| Stage | Function | Why |
|---|---|---|
| Bandpass 20–450 Hz | 4th-order Butterworth, SOS form | Blumenthal et al. (2005) committee report range: 20 Hz removes DC drift and movement artefacts; 450 Hz cuts mains-related high-frequency noise. |
| **Hampel outlier replacement** | 50-sample window, 3·MAD rule | Replaces transient spike artefacts (electrode contact, cable tug) with the local rolling median. *Order matters — Hampel must come **after** the bandpass; otherwise high-frequency noise inflates the local MAD and the spike threshold lifts.* |
| Full-wave rectification | $\lvert x \rvert$ | Converts bipolar EMG to an activation-magnitude proxy. |
| Low-pass envelope | 10 Hz, 2nd-order Butterworth | Smooths the rectified train into the slow activation envelope used by psychophysiology. |

**Output:** `processed_data/event_windows_all.pkl` (consumed by every downstream script via Stage 2).

**Cost:** Re-running this step on the full cohort takes **~20–30 minutes** on a laptop. It is run once and cached.

*The source file is `run_pipeline.py` in this folder. Open it directly, or run the cell below to see the EMG pipeline excerpt.*

In [ ]:
# Show the EMG pipeline excerpt (the core of run_pipeline.py).
from inspect import getsource
import run_pipeline
print('--- butterworth_bandpass ---')
print(getsource(run_pipeline.butterworth_bandpass))
print('--- hampel_filter ---')
print(getsource(run_pipeline.hampel_filter))
print('--- emg_envelope ---')
print(getsource(run_pipeline.emg_envelope))

### Saliency taxonomy

The dataset annotates every stimulus with a numeric `EventID`. I hand-classified each ID as **high-saliency** (intense emotional set-piece) or **low-saliency** (ambient prop). This taxonomy is **load-bearing**: every saliency target in the thesis flows from it.

| Scene | High-saliency IDs | Low-saliency IDs |
|---|---|---|
| Positive | 1, 2, 3, 6, 7, 202 | 4, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 21, 23 |
| Negative | 32, 33, 34, 35, 37, 38, 39, 52, 53, 54, 56 | 30, 31, 36, 40, 41, 43, 44, 46, 47, 48, 49, 51, 55, 62 |

## 1B. `extract_spectral_features.py` — windows → 459-dim feature bank

**What it does.** For every event window from Stage 1A, computes a **459-dimensional feature vector** by combining a 17-feature bank per channel × 9 channels × 3 segments (pre-event, post-event, full).

**The 17 features per channel:**

| Group | Features |
|---|---|
| Time-domain (9) | RMS, MAV (mean absolute value), variance, integrated EMG, waveform length, zero crossings, slope-sign changes, skewness, kurtosis |
| Frequency-domain (8) | mean power freq, median power freq, peak freq, relative band power in 20–80 / 80–150 / 150–250 / 250–450 Hz, spectral entropy |

**Why three segments?** The pre-event segment is the per-window baseline (necessary for normalisation); the post-event segment is the response; the full segment is what the classifiers actually train on. Computing all three lets downstream scripts experiment with baseline-corrected vs raw representations without re-extracting features.

**Why these features?** This is the canonical surface-EMG pattern-recognition feature bank (Phinyomark et al., 2013). It is **only used by `train_spectral.py`** — the other training scripts use the simpler 10 Hz envelope directly.

**Output:** `processed_data/event_windows_spectral.pkl` (Stage 1A windows + per-event 459-feature vector).

---
# Stage 2 — Delay extraction (the headline result)

## 2A. `estimate_delays.py` — per-event Delay A, B, C

This is the script that produces the **1.4-second cognitive–motor gap** result that anchors the thesis. It is a **rule-based algorithm** — no machine learning, no fitting, no random seed; the same data always produces the same numbers.

### Three latencies, three definitions

Let $T_0$ = the moment a stimulus's EventID first appears in the active set.

| Latency | Definition | Validity gate | Literature |
|---|---|---|---|
| **Delay A** — psychophysiological onset | First sample at which **any of the 9 EMG channels** crosses its pre-event baseline ($\mu$ + 3$\sigma$, computed over the 2 s before $T_0$). Aggregate across channels by **minimum** — *the first muscle to fire.* | 100–500 ms | Dimberg & Thunberg (1998) facial-EMG literature |
| **Delay C** — total observable delay | First sample at which the **joystick valence** signal moves significantly from its pre-event baseline. Two rules, reported side-by-side: (i) amplitude threshold $\lvert V - \mu \rvert > 3\sigma$; (ii) **slope rule** $\lvert dV/dt \rvert > 0.1$ unit/s. | 500–5000 ms | Huang et al. (2015) conscious-response window |
| **Delay B** — cognitive–motor gap | $\text{Delay B} = \text{Delay C} - \text{Delay A}$. Only computed when both A and C are valid. | 100–4900 ms, positive only | Derived |

### Why a 10 ms skip after $T_0$?

The Delay A detector starts its search at $T_0 + 10$ ms rather than $T_0$ itself. Stimulus-simultaneous artefacts (e.g. a sudden sound triggering a startle in the same sample as the EventID appears) would otherwise register as a spurious 0 ms delay.

### Why "minimum across channels" for Delay A?

The Dimberg literature defines facial-EMG onset as the latency of *any* muscle response, not the aggregate of all muscles. A smile activates the zygomaticus; a frown activates the corrugator. Taking the minimum across channels captures "the first muscle that fired", which is what the literature measures.

### Why the slope rule for Delay C? 

**Why slope and not just amplitude.** Continuous self-reports do not always cross an amplitude threshold cleanly. A participant who slowly tilts the joystick from $0$ to $-0.3$ over two seconds may never exceed a 3$\sigma$ amplitude band, yet the *moment of intent* — the instant their hand started moving — is well-defined. The first derivative $dV/dt$ captures that instant: it spikes the moment the rating changes, regardless of how large the eventual change is. Delay C is supposed to mark when the participant's conscious appraisal **began**, not when it had finished, which is why a slope-based criterion is more faithful to the construct than a pure amplitude rule.

**How it is computed.** For each post-event sample, the script computes $\lvert dV/dt \rvert$ using forward differences and takes the **first sample to exceed 0.1 valence-units per second**, evaluated at midpoint time between adjacent samples (the derivative is naturally located between the two samples it spans). A floor of $10^{-6}$ on $\Delta t$ guards against ratings sampled at zero-width intervals.

**What this is based on.** The slope-detection idea is standard in continuous-rating affective-computing pipelines (Mavridou et al., 2025; Soleymani et al., 2012, who use derivative-based change detection on continuous ratings to align EMG/EEG with conscious response). The choice of $0.1$ units/s as the cut-off is empirically anchored: at 10 Hz sampling, $0.1$ units/s corresponds to a perceptible joystick deflection of $0.01$ unit per sample — small enough to catch the onset of a deliberate movement, large enough to reject the participant's natural baseline tremor (which typically sits below $0.03$ units/s in this dataset).

**Why we are sure it is the right approach.** The script does **not rely on the slope rule alone**. For every event it reports both the amplitude-threshold rule and the slope rule, and emits the RMSE between the two at the end of its run. On the cohort, the RMSE is in the order of a few hundred milliseconds — the two rules disagree about *exactly when* the rating changed, but they agree about *whether* it changed, and the per-event differences average out at the cohort level. The thesis reports the threshold rule as the primary measure (because it is the more conservative, higher-precision detector) and the slope rule as a robustness check (because it has higher recall on gradual ratings). The 1.4 s headline result is computed on the threshold rule but is essentially unchanged if the slope rule is substituted: $\mu_{\text{slope}}(\text{Delay C})$ differs from $\mu_{\text{threshold}}(\text{Delay C})$ by less than 200 ms across the cohort. **This dual-rule reporting is the script's main defence against the criticism that any single change-detection rule is arbitrary.**

In [ ]:
# Load the per-event delay table that estimate_delays.py produced.
delays = pd.read_csv(RESULTS / 'delays_per_event.csv')
print(f'{len(delays)} events from {delays["pid"].nunique()} participants\n')

for col, vcol, lbl in [
    ('delay_a_ms', 'a_valid', 'Delay A  (EMG onset)'),
    ('delay_c_ms', 'c_valid', 'Delay C  (joystick onset)'),
    ('delay_b_ms', 'b_valid', 'Delay B  (cognitive-motor gap, C - A)'),
]:
    v = delays.loc[delays[vcol], col]
    print(f'{lbl:42s} n={len(v):>4d}  median={v.median():.0f} ms  mean={v.mean():.0f} ms')

print('\n→ Delay B median ≈ 1.4 s. The 200 ms heuristic captures only ~12% of this.')

In [ ]:
# The corresponding figures used in §5.2 of the thesis.
for fname in ['delays_overall_distribution.png', 'delays_a_vs_c_scatter.png', 'delays_by_saliency.png']:
    p = PLOTS / fname
    if p.exists():
        display(Markdown(f'### `{fname}`'))
        display(Image(filename=str(p), width=900))

---
# Stage 3 — Statistics

## 3A. `analyze_delays.py` — distributions and saliency contrast

Reads `delays_per_event.csv` and produces three figures:

1. Overall histograms of Delay A, C, B with mean and median annotations (the figure that visualises the 227 / 1717 / 1379 ms headline).
2. Same delays split by high- vs low-saliency, with Mann–Whitney U statistic on each panel.
3. Per-event Delay A vs Delay C scatter, with $y = x$ reference.

Saliency contrast (Mann–Whitney U, event-level): high vs low saliency yields a directional but non-significant difference at the BH-corrected level (consistent with the modulation result below).

## 3B. `analyze_modulation.py` — per-subject modulation battery

Added after the May 1 supervisor meeting. Aggregates `delays_per_event.csv` to per-participant medians and joins with the participant-metadata Excel (questionnaires, demographics, presence sub-scales, Big Five, DASS, empathy, alexithymia, free-recall memory). Runs **44 tests**:

* Mann–Whitney: positive vs negative valence (event- and subject-level) on each delay
* Mann–Whitney: high vs low saliency on each delay
* Mann–Whitney: gender, Active vs Passive condition on Delay B
* Spearman ρ: every continuous predictor vs per-subject Delay A and Delay B

All p-values are jointly **Benjamini–Hochberg corrected**.

**Result.** No test survives BH correction at $\alpha = 0.05$. The strongest directional trend is +141 ms longer Delay B for positive vs negative scenes ($p_{\text{raw}} = 0.022$). Reported as exploratory in §6.4.

**Why this is not failure.** The matched-questionnaire subset is small ($n = 35$ with full personality data; $n = 118$ with valid Delay B medians but only $35$ also have the metadata). The result is consistent with two interpretations: (a) the delay genuinely does not vary at the individual level, or (b) the sample is too small to detect what variation exists. The thesis prefers (b).

In [ ]:
# Load the modulation stats and show the top 5 raw-p hits.
mod = pd.read_csv(RESULTS / 'delays_modulation_stats.csv').sort_values('p_raw')
print(f'{len(mod)} tests; BH-corrected significant at α=0.05:', (mod["p_bh"] < 0.05).sum())
mod[['test', 'factor', 'target', 'n', 'statistic', 'p_raw', 'p_bh', 'sig_bh', 'direction']].head(5)

In [ ]:
# Modulation summary plot (cited as Figure in §5.2.4).
for fname in ['delay_b_modulation_summary.png']:
    p = PLOTS / fname
    if p.exists():
        display(Image(filename=str(p), width=900))

---
# Stage 4 — Models

Six training scripts, each answering a different sub-question. None of them are re-executed in this notebook — the cached metrics in `results/` already contain the numbers reported in the thesis.

All trainers share three conventions:

* **Subject-level 80/10/10 train/val/test split** with `SEED = 42`. Each participant appears in exactly one split — no within-subject leakage.
* Features come from `processed_data/multimodal_windows.npz` or `event_windows_spectral.pkl`, both produced by Stage 1.
* Metrics are saved as JSON under `results/metrics_*.json`.

## 4A. `train_lstm_v4_full.py` — BiLSTM vs RF (the SQ1 head-to-head)

**Question.** Can a bidirectional LSTM seq2seq, given multi-band EMG plus physiology plus scene context, beat a Random Forest that uses the static 200 ms lag from Mavridou et al. (2025)?

**Model variants (rows of the result table):**

| ID | Model | Features |
|---|---|---|
| A | RF, 200 ms static lag | multi-band EMG (45) |
| B | BiLSTM seq2seq, CCC loss | multi-band EMG (45) |
| C | BiLSTM seq2seq, CCC loss | multi-band EMG + physiology (59) |
| D | BiLSTM seq2seq, CCC loss | multi-band EMG + physiology + scene (60) |

**Why CCC loss.** Concordance correlation coefficient (Lin, 1989) is the standard metric in continuous affect recognition and penalises both correlation **and** scale/offset bias. Training directly on $1 - \text{CCC}$ (rather than on MSE) maximises what we ultimately report.

**Why baseline-corrected valence target.** Each window's pre-event mean is subtracted from its valence target. Without this step, between-subject offset bias dominates the cross-subject CCC — different participants use different parts of the $[-1, +1]$ scale, so the model learns to predict individual offsets rather than within-window dynamics.

**Result.** RF wins on **12 of 13 test subjects**, Wilcoxon signed-rank $p = 0.0007$. Reported honestly in §5.3 and §6.2.

**Why didn't the LSTM beat RF?** Subject-variance ceiling. Different people's bodies react differently to the same stimulus, and no amount of model capacity overcomes a per-subject signal-to-noise ratio that low. The bottleneck is the **cross-subject signal**, not the architecture.

In [ ]:
with open(RESULTS / 'metrics_sq1_v4.json') as f:
    sq1 = json.load(f)

print('--- Headline numbers from metrics_sq1_v4.json ---')
print(json.dumps({k: v for k, v in sq1.items() if k in ('summary', 'wilcoxon', 'rf_vs_lstm', 'headline')}, indent=2)[:2500])

p = PLOTS / 'sq1_v4_comparison.png'
if p.exists():
    display(Image(filename=str(p), width=900))

## 4B. `train_lstm_multiband.py` — earlier v3 variant

The v3 script that immediately preceded v4. It introduces multi-band EMG features (45 per timestep, vs the 9 of v2) but does **not** yet add physiology, scene-label fusion, or baseline-corrected target. Kept in the submission because Figure 5 of the thesis (the per-subject CCC scatter) is rendered from this script's output.

## 4C. `train_classification.py` — binary valence-direction classifier (Section 5.3.2)

Reframes SQ1 as a clean binary task to obtain interpretable accuracy: per event window, predict whether the joystick valence will **rise or fall** post-event. The model grid spans majority-class baseline → scene-only logistic → RF on summary features → BiLSTM sequence classifier, with three feature-set conditions each. Used to give the supervisor an accuracy % rather than a CCC value.

## 4D. `train_spectral.py` — the saliency AUC = 0.729 result

**Question.** *Once we score the right target*, does the EMG signal carry any information?

Uses the 459-dim spectral feature bank from Stage 1B. Three feature conditions: EMG-spectral only, scene-only (the confound), and scene + EMG-spectral combined. Two targets: continuous valence regression and binary saliency classification.

**Result.** EMG **does** carry information — but for **saliency** (arousal proxy), not for **signed valence**. The headline number: **AUC = 0.729** on saliency with EMG + scene. The reframing of the thesis narrative ("EMG indexes arousal, not valence") flows from this script.

In [ ]:
with open(RESULTS / 'metrics_spectral.json') as f:
    sp = json.load(f)
print(json.dumps(sp, indent=2)[:2200])

## 4E. `train_svm_variations.py` — SVM × 4 timing windows × 3 feature sets (Section 5.3.3)

**Supervisor-driven ablation.** Two questions came out of the May 1 meeting:

1. *Where in the response is saliency information concentrated?* The empirical $\text{Delay B} = 1379$ ms suggests a late-appraisal window may be more informative than the event-locked window.
2. *Does an SVM equal or exceed an LSTM at this task?* (Supervisor's prior, based on her own experience.)

**Four timing windows** (from the supervisor's whiteboard sketch):

| Variation | Window | Position |
|---|---|---|
| V1 — event-locked | $T_0 \to T_0 + 500$ ms | the default |
| V2 — mid-appraisal | $T_0 + B/2 \to T_0 + B/2 + 500$ ms | $\approx T_0 + 700$ ms |
| V3 — late-appraisal | $T_0 + B \to T_0 + B + 500$ ms | $\approx T_0 + 1400$ ms ≈ Delay C onset |
| Baseline | $T_0 + 200 \to T_0 + 700$ ms | the Mavridou 200 ms heuristic |

Per-channel features: mean, median, MAV, standard deviation. 9 channels × 4 stats = 36 features for the broadband condition.

**Result.** SVM-RBF and RF **tie at AUC 0.705** (multi-band + scene). The timing window does not matter — V1/V2/V3/baseline all land within 0.02 AUC of each other. EMG alone (without scene context) is at chance regardless of window.

**What this means.** Saliency information is **not** time-locked to a specific post-event interval — the EMG signal carries cohort-level saliency information broadly across the response, but the scene label dominates. The supervisor's SVM prior was confirmed; the late-appraisal hypothesis was not.

In [ ]:
with open(RESULTS / 'metrics_window_variations.json') as f:
    wv = json.load(f)

# Compact table.
rows = []
for v, vres in wv.items():
    for fset, fres in vres['feature_sets'].items():
        for mname, r in fres['models'].items():
            rows.append({'variation': v, 'features': fset, 'model': mname, 'AUC': round(r['auc'], 3), 'Acc%': round(r['acc']*100, 1)})
pd.DataFrame(rows).pivot_table(index=['variation', 'features'], columns='model', values='AUC')

## 4F. `train_spectral_slope.py` — shape-feature future-work starting point (Section 6.4)

Exploratory addition referenced by name in the thesis at §6.4. Augments the 459-dim spectral bank with per-channel **linear slopes** of the 10 Hz EMG envelope on the pre / post / full segments (27 extra features → 486 total). The supervisor's methodological point was that *mean*-based features may be poor proxies for **valence**-specific muscle activation, because it is the **shape** of the envelope (slope of activation onset, area under the curve) that distinguishes a smile from a grimace, not the average amplitude. This script is the concrete starting point of that future-work direction; the marginal gain in the current small test set is modest, but the shape-feature direction is the recommended next step in §6.4.

---
# Reproduce

The dataset (Mavridou et al., 2025) is **not** redistributed with this submission. Place the raw data under `All Data/<participant_id>/` (one folder per participant; each folder must contain `*_POSITIVE_03_*_raw.txt`, `*_POSITIVE_03_*.json`, `*_negative_03_*_raw.txt`, `*_negative_03_*.json`, and the corresponding `Rating_Scene_*` files).

Then run, in order:

```bash
# Stage 1 — preprocessing (~20–30 min on a laptop)
python run_pipeline.py
python extract_spectral_features.py

# Stage 2 — delay extraction (~5–10 min)
python estimate_delays.py

# Stage 3 — statistics (seconds)
python analyze_delays.py
python analyze_modulation.py

# Stage 4 — models (~1–4 h on a laptop with MPS / CUDA)
python train_lstm_v4_full.py
python train_lstm_multiband.py
python train_classification.py
python train_spectral.py
python train_svm_variations.py
python train_spectral_slope.py
```

All trainers use `SEED = 42` and a subject-level 80/10/10 split. Re-running any single script with the same data reproduces the numbers reported in the thesis.

**Software environment.** Python 3.11, NumPy 1.26, SciPy 1.11, pandas 2.1, scikit-learn 1.4, PyTorch 2.2 with Apple Metal Performance Shaders (`mps`) device backend, Matplotlib 3.8.